# İşaret Dili — Mini Kelime Tanıma (AUTSL, 3 kelime + "diğer")

Bu notebook baştan sona şunları yapar:

1. AUTSL videolarını ve etiketlerini bulur, seçtiğin kelimelerin videolarını ayırır.
2. **Görüntü işleme:** MediaPipe ile her videonun her karesinden el ve vücut noktalarını çıkarır.
3. **Makine öğrenmesi:** Küçük bir GRU modelini eğitir ve modelin hiç görmediği kişilerle test eder.
4. Kendi bilgisayarında kamerayla çalışacak demoyu `demo_paketi.zip` olarak hazırlar.

**Çalıştırmadan önce (sağ panel):**
- *Settings → Internet:* **On** (MediaPipe'ı ve model dosyalarını indirmek için; telefon doğrulaması isteyebilir)
- *Settings → Accelerator:* None yeterli, GPU gerekmez
- *Input:* AUTSL veri setin ekli olmalı

Sonra **Run All**. Bitince sağdaki *Output* bölümünden `demo_paketi.zip` dosyasını indir.

> Kaggle senin kameranı açamaz; notebook uzaktaki bir sunucuda çalışır. Kamera demosu zip'in içindeki `app.py` ile kendi bilgisayarında çalışır.

## 1. Ayarlar
Sadece bu hücreyi değiştirmen yeterli.

In [ ]:
SECILEN_KELIMELER = ["anne", "doktor", "evet"]   # 2-3 kelime; listede yoksa en yakın ad kullanılır
DIGER_SINIFI = True      # True: başka kelimelerden "diğer" sınıfı eklenir; demoda yanlış alarmı azaltır
GORUNEN_AD = {}          # ekranda farklı yazılsın istersen, örnek: {"tesekkur": "teşekkür"}
ESIK = 0.6               # canlı demoda bu güvenin altındaki tahminler kabul edilmez
T = 48                   # her örnek bu kadar kareye getirilir
EPOCH = 80               # en fazla bu kadar tur eğitilir (iyileşme durursa erken biter)
VERI_KLASORU = "/kaggle/input"
CIKTI = "/kaggle/working"

## 2. Kurulum ve MediaPipe model dosyaları

In [ ]:
!pip install -q mediapipe==0.10.32
import os, re, sys, glob, json, time, random, difflib, shutil, urllib.request, multiprocessing
import numpy as np, pandas as pd, cv2, torch
import matplotlib.pyplot as plt
import mediapipe as mp

os.chdir(CIKTI)
sys.path.insert(0, CIKTI)          # birazdan yazacağımız .py dosyaları buradan içe aktarılacak
print("mediapipe", mp.__version__, "| opencv", cv2.__version__, "| torch", torch.__version__)

In [ ]:
MODEL_URL = {
    "hand_landmarker.task": "https://storage.googleapis.com/mediapipe-models/hand_landmarker/hand_landmarker/float16/latest/hand_landmarker.task",
    "pose_landmarker_full.task": "https://storage.googleapis.com/mediapipe-models/pose_landmarker/pose_landmarker_full/float16/latest/pose_landmarker_full.task",
}
for ad, url in MODEL_URL.items():
    if not os.path.exists(ad):
        try:
            urllib.request.urlretrieve(url, ad)
        except Exception as e:
            raise RuntimeError("Model indirilemedi. Settings -> Internet açık mı?") from e
    print(ad, round(os.path.getsize(ad) / 1e6, 1), "MB")

## 3. Veri setini bul
Videoları, etiket dosyalarını ve kelime listesini klasör adlarına bakmadan otomatik bulur.

In [ ]:
# 3a) Renkli videolar (derinlik videoları kullanılmaz)
def ornek_adi(yol):
    return re.sub(r"(_color)?\.mp4$", "", os.path.basename(yol), flags=re.I)

tum_mp4 = glob.glob(f"{VERI_KLASORU}/**/*.mp4", recursive=True)
VIDEO = {}
for p in sorted(tum_mp4):
    if not p.lower().endswith("_depth.mp4"):
        VIDEO.setdefault(ornek_adi(p), p)
print(len(tum_mp4), "mp4 dosyası,", len(VIDEO), "renkli video")
assert VIDEO, "Hiç video bulunamadı. Sağ panelden AUTSL'yi 'Add Input' ile eklediğinden emin ol."

# 3b) Etiket dosyaları: "signerX_sampleY,etiket" satırları içeren tüm CSV'ler
csvler = sorted(glob.glob(f"{VERI_KLASORU}/**/*.csv", recursive=True))
ETIKET = {}
for f in csvler:
    try:
        d = pd.read_csv(f, header=None, dtype=str, on_bad_lines="skip")
    except Exception:
        continue
    if d.shape[1] < 2:
        continue
    ad = d.iloc[:, 0].astype(str).map(lambda s: re.sub(r"(_color)?(\.mp4)?$", "", os.path.basename(s.strip())))
    no = pd.to_numeric(d.iloc[:, 1], errors="coerce")
    uygun = ad.str.match(r"^signer\d+_sample\d+$") & no.notna()
    if uygun.sum() > 0:
        ETIKET.update(dict(zip(ad[uygun], no[uygun].astype(int))))
        print(f"etiket dosyası: {f} -> {uygun.sum()} satır")
print("Toplam etiketli örnek:", len(ETIKET))
if not ETIKET:
    for f in csvler[:5]:
        print("-----", f); print(open(f, encoding="utf-8", errors="ignore").read(300))
    raise RuntimeError("Etiket dosyası bulunamadı; yukarıdaki CSV'lerin ilk satırlarını kontrol et.")

# 3c) Kelime listesi (etiket no -> kelime): ~226 satırlı, ilk sütunu 0,1,2,... olan CSV
SINIF = {}
for f in csvler:
    for baslik in (0, None):
        try:
            d = pd.read_csv(f, header=baslik)
        except Exception:
            continue
        if not (100 <= len(d) <= 400) or d.shape[1] < 2:
            continue
        no = pd.to_numeric(d.iloc[:, 0], errors="coerce")
        if no.notna().all() and set(no.astype(int)) == set(range(len(d))):
            kolon = next((c for c in d.columns if str(c).strip().upper() == "TR"), d.columns[1])
            SINIF = {int(n): str(w).strip() for n, w in zip(no, d[kolon])}
            print("kelime listesi:", f, "| kelime sütunu:", kolon)
            break
    if SINIF:
        break
if not SINIF and len(set(ETIKET.values())) == 226:
    # Bazı Kaggle kopyalarında kelime listesi dosyası yok; AUTSL'nin resmi listesi (SignList_ClassId_TR_EN) kullanılır.
    RESMI_LISTE = """abla acele acikmak afiyet_olsun agabey agac agir aglamak aile akilli akilsiz akraba alisveris anahtar anne arkadas ataturk ayakkabi ayna ayni baba bahce bakmak bal bardak bayrak bayram bebek bekar beklemek ben benzin beraber bilgi_vermek biz calismak carsamba catal cay caydanlik cekic cirkin cocuk corba cuma cumartesi cuzdan dakika dede degistirmek devirmek devlet doktor dolu dugun dun dusman duvar eczane eldiven emek emekli erkek et ev evet evli ezberlemek fil fotograf futbol gecmis gecmis_olsun getirmek gol gomlek gormek gostermek gulmek hafif hakli hali hasta hastane hata havlu hayir hayirli_olsun hayvan hediye helal hep hic hoscakal icmek igne ilac ilgilenmemek isik itmek iyi kacmak kahvalti kalem kalorifer kapi kardes kavsak kaza kemer keske kim kimlik kira kitap kiyma kiz koku kolonya komur kopek kopru kotu kucak leke maas makas masa masallah melek memnun_olmak mendil merdiven misafir mudur musluk nasil neden nerede nine ocak oda odun ogretmen okul olimpiyat olmaz olur onlar orman oruc ozur_dilemek pamuk pantolon para pastirma patates pazar pazartesi pencere persembe piknik polis psikoloji rica_etmek saat sabun salca sali sampiyon sapka savas seker selam semsiye sen senet serbest ses sevmek seytan sinir siz soylemek soz sut tamam tarak tarih tatil tatli tavan tehlike telefon terazi terzi tesekkur tornavida turkiye turuncu tuvalet un uzak uzgun var vergi yakin yalniz yanlis yapmak yarabandi yardim yarin yasak yastik yatak yavas yemek yemek_pisirmek yildiz yok yol yorgun yumurta zaman zor""".split()
    SINIF = dict(enumerate(RESMI_LISTE))
    print("Kelime listesi dosyası yok; AUTSL'nin resmi kelime listesi kullanılıyor.")
if not SINIF:
    print("UYARI: kelime listesi bulunamadı; kelimeleri numarayla seç (örn. SECILEN_KELIMELER = ['14', '52']).")
    SINIF = {i: str(i) for i in sorted(set(ETIKET.values()))}
print(len(SINIF), "kelime:", ", ".join(SINIF[i] for i in sorted(SINIF)))

## 4. Kelimeleri seç ve kişi bazında böl
Test setindeki kişiler eğitimde hiç bulunmaz; sonuç bu yüzden gerçek kullanımı yansıtır.

In [ ]:
def sade(s):   # karşılaştırma için: küçük harf, Türkçe harfler sade
    s = str(s).replace("İ", "i").replace("I", "ı").lower().strip()
    return s.translate(str.maketrans("çğıöşü", "cgiosu")).replace(" ", "_")

ad_no = {sade(w): i for i, w in SINIF.items()}
secilen = []
for k in SECILEN_KELIMELER:
    if sade(k) in ad_no:
        secilen.append(ad_no[sade(k)])
        continue
    yakin = difflib.get_close_matches(sade(k), list(ad_no), n=3, cutoff=0.6)
    if not yakin:
        raise ValueError(f"'{k}' kelime listesinde yok. Yukarıdaki listeden bir kelime seç.")
    print(f"'{k}' listede yok, en yakın kelime kullanılıyor: {yakin[0]}  (diğer öneriler: {yakin[1:]})")
    secilen.append(ad_no[yakin[0]])
secilen = list(dict.fromkeys(secilen))
assert len(secilen) >= 2, "En az 2 farklı kelime seçmelisin."

tum = pd.DataFrame([(s, n, VIDEO[s]) for s, n in ETIKET.items() if s in VIDEO], columns=["sample", "label", "path"])
tum["signer"] = tum["sample"].str.extract(r"signer(\d+)_")[0].astype(int)
print("Etiketi ve videosu olan örnek:", len(tum), "| işaretçi sayısı:", tum.signer.nunique())

SINIF_ADLARI = [GORUNEN_AD.get(SINIF[i], SINIF[i]) for i in secilen]
veri = tum[tum.label.isin(secilen)].copy()
veri["y"] = veri.label.map({n: k for k, n in enumerate(secilen)})
if DIGER_SINIFI:
    kalan = tum[~tum.label.isin(secilen)]
    adet = min(len(kalan), int(veri.groupby("y").size().mean()))
    diger = kalan.sample(n=adet, random_state=42).copy()
    diger["y"] = len(secilen)
    veri = pd.concat([veri, diger])
    SINIF_ADLARI.append("(diğer)")
veri = veri.reset_index(drop=True)

from sklearn.model_selection import GroupShuffleSplit
a, test_i = next(GroupShuffleSplit(n_splits=1, test_size=0.15, random_state=42).split(veri, groups=veri.signer))
egitdog = veri.iloc[a]
b, dog_i = next(GroupShuffleSplit(n_splits=1, test_size=0.18, random_state=42).split(egitdog, groups=egitdog.signer))
veri["split"] = "train"
veri.loc[egitdog.index[dog_i], "split"] = "val"
veri.loc[veri.index[test_i], "split"] = "test"
kisiler = {s: set(veri[veri.split == s].signer) for s in ["train", "val", "test"]}
assert kisiler["train"].isdisjoint(kisiler["val"]) and kisiler["train"].isdisjoint(kisiler["test"]) \
    and kisiler["val"].isdisjoint(kisiler["test"]), "Aynı kişi iki bölmede olamaz!"

tablo = pd.crosstab(veri.y.map(dict(enumerate(SINIF_ADLARI))), veri.split).reindex(
    columns=["train", "val", "test"], fill_value=0)
print({s: len(k) for s, k in kisiler.items()}, "kişi (eğitim / doğrulama / test)")
print(tablo)

### Seçilen kelimelerin örnek videoları
Kamera demosunda bu işaretleri sen yapacaksın; buradan izleyip çalış. Her kelimeden iki farklı kişi gösterilir.
Videodaki kişi **sağ elini** kullanıyorsa sen de **sağ elini** kullan (karşındaki birini taklit eder gibi, ayna gibi değil).
Videolar `ornek_videolar/` klasörüne de kaydedilir ve demo paketine eklenir.

In [ ]:
import io, base64
from PIL import Image
from IPython.display import HTML, display

def video_gif(yol, boyut=256, adim=2):
    """Videoyu, notebook'ta ve her bilgisayarda oynayan hareketli GIF'e çevirir."""
    cap = cv2.VideoCapture(yol)
    fps = cap.get(cv2.CAP_PROP_FPS) or 30
    kareler, i = [], 0
    while True:
        ok, kare = cap.read()
        if not ok:
            break
        if i % adim == 0:
            kareler.append(Image.fromarray(cv2.cvtColor(cv2.resize(kare, (boyut, boyut)), cv2.COLOR_BGR2RGB)))
        i += 1
    cap.release()
    buf = io.BytesIO()
    kareler[0].save(buf, format="GIF", save_all=True, append_images=kareler[1:],
                    duration=int(1000 * adim / fps), loop=0)
    return buf.getvalue()

os.makedirs("ornek_videolar", exist_ok=True)
html = ""
for k in range(len(secilen)):
    ad = SINIF_ADLARI[k]
    html += f"<h4 style='margin:12px 0 4px'>{ad}</h4>"
    for _, r in veri[veri.y == k].drop_duplicates("signer").head(2).iterrows():
        gif = video_gif(r.path)
        with open(f"ornek_videolar/{sade(ad)}_{r['sample']}.gif", "wb") as f:
            f.write(gif)
        html += (f"<div style='display:inline-block;margin:4px;text-align:center'>"
                 f"<img src='data:image/gif;base64,{base64.b64encode(gif).decode()}' width='256'>"
                 f"<br><small>{r['sample']}</small></div>")
display(HTML(html))

## 5. Görüntü işleme: nokta çıkarma kodu
Bu dosyalar hem burada hem kamera demosunda **aynen** kullanılır.

In [ ]:
%%writefile landmarks.py
"""Videodan / kameradan el ve vücut noktalarını çıkarma (görüntü işleme kısmı).

Her kare 201 sayılık bir vektöre dönüşür:
  0-74    vücut noktaları 0-24 (belden yukarısı), her biri x, y, z
  75-137  sol el, 21 nokta
  138-200 sağ el, 21 nokta
Bulunamayan parçanın yeri 0 ile doldurulur.
"""
import os

import cv2
import numpy as np
import mediapipe as mp
from mediapipe.tasks import python as mp_python
from mediapipe.tasks.python import vision

N_POSE = 25                       # vücut noktaları 0-24
N_FEAT = (N_POSE + 21 + 21) * 3   # = 201


def make_landmarkers(hand_model="hand_landmarker.task", pose_model="pose_landmarker_full.task"):
    """El ve vücut nokta bulucularını VIDEO modunda oluşturur."""
    hand_opts = vision.HandLandmarkerOptions(
        base_options=mp_python.BaseOptions(model_asset_path=hand_model),
        running_mode=vision.RunningMode.VIDEO,
        num_hands=2,
        min_hand_detection_confidence=0.5,
        min_hand_presence_confidence=0.5,
        min_tracking_confidence=0.5)
    pose_opts = vision.PoseLandmarkerOptions(
        base_options=mp_python.BaseOptions(model_asset_path=pose_model),
        running_mode=vision.RunningMode.VIDEO,
        num_poses=1)
    return (vision.HandLandmarker.create_from_options(hand_opts),
            vision.PoseLandmarker.create_from_options(pose_opts))


def to_vector(pose_res, hand_res):
    """Bir karenin MediaPipe sonuçlarını 201 sayılık vektöre çevirir."""
    pose = np.zeros((N_POSE, 3), np.float32)
    left = np.zeros((21, 3), np.float32)
    right = np.zeros((21, 3), np.float32)
    has_pose = len(pose_res.pose_landmarks) > 0
    if has_pose:
        lms = pose_res.pose_landmarks[0][:N_POSE]
        pose = np.array([[p.x, p.y, p.z] for p in lms], np.float32)
    for hand, handed in zip(hand_res.hand_landmarks, hand_res.handedness):
        pts = np.array([[p.x, p.y, p.z] for p in hand], np.float32)
        if has_pose:
            # Elin bileği, vücudun sol (15) / sağ (16) bileğinden hangisine yakınsa o eldir.
            d_left = np.linalg.norm(pts[0, :2] - pose[15, :2])
            d_right = np.linalg.norm(pts[0, :2] - pose[16, :2])
            is_left = d_left < d_right
        else:
            is_left = handed[0].category_name == "Left"
        if is_left:
            left = pts
        else:
            right = pts
    return np.concatenate([pose, left, right]).reshape(-1)


def to_mp_image(frame_bgr):
    """OpenCV karesini (BGR) MediaPipe görüntüsüne (RGB) çevirir."""
    rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
    return mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)


def extract_video(path, hand_model="hand_landmarker.task", pose_model="pose_landmarker_full.task"):
    """Bir videoyu (kare_sayısı, 201) boyutunda diziye çevirir."""
    hand_lm, pose_lm = make_landmarkers(hand_model, pose_model)   # her video için yeni: zaman damgası 0'dan başlar
    cap = cv2.VideoCapture(path)
    fps = cap.get(cv2.CAP_PROP_FPS) or 30
    frames, i = [], 0
    try:
        while True:
            ok, frame = cap.read()
            if not ok:
                break
            img = to_mp_image(frame)
            ts = int(i * 1000 / fps)              # milisaniye; her karede artmak zorunda
            frames.append(to_vector(pose_lm.detect_for_video(img, ts),
                                    hand_lm.detect_for_video(img, ts)))
            i += 1
    finally:
        cap.release()
        hand_lm.close()
        pose_lm.close()
    return np.array(frames, np.float32).reshape(-1, N_FEAT)


def process_one(sample, path, out_dir):
    """Tek videoyu işleyip .npy kaydeder. Dosya zaten varsa atlar (kaldığı yerden devam)."""
    out_path = os.path.join(out_dir, sample + ".npy")
    if os.path.exists(out_path):
        return sample, "atlandı"
    try:
        seq = extract_video(path)
        np.save(out_path, seq.astype(np.float16))
        return sample, len(seq)
    except Exception as e:
        return sample, "HATA: " + str(e)

In [ ]:
%%writefile preprocess.py
"""Nokta dizilerini modele hazırlama. Eğitimde ve canlı demoda AYNI dosya kullanılır."""
import numpy as np

N_PTS = 67    # 25 vücut + 21 sol el + 21 sağ el


def trim(seq):
    """Başta ve sonda hiç el görünmeyen kareleri atar."""
    has_hand = (seq[:, 75:] != 0).any(axis=1)
    idx = np.where(has_hand)[0]
    return seq if len(idx) == 0 else seq[idx[0]: idx[-1] + 1]


def random_time_crop(seq, rng=np.random):
    """Sadece eğitimde: dizinin rastgele %80-100'lük bir parçasını alır."""
    n = len(seq)
    if n < 10:
        return seq
    keep = int(n * rng.uniform(0.8, 1.0))
    start = rng.randint(0, n - keep + 1)
    return seq[start: start + keep]


def resample(seq, t):
    """Diziyi tam t kareye getirir: kısaysa kareleri tekrarlar, uzunsa seyreltir."""
    if len(seq) == 0:
        return np.zeros((t, N_PTS * 3), np.float32)
    idx = np.linspace(0, len(seq) - 1, t).round().astype(int)
    return seq[idx]


def normalize(seq):
    """Omuz ortasını merkeze alır, omuz genişliğini 1 birim yapar. Eksik noktalar 0 kalır."""
    s = seq.reshape(len(seq), N_PTS, 3).astype(np.float32).copy()
    missing = (s == 0).all(axis=2)
    ok = ~missing[:, 11] & ~missing[:, 12]          # iki omzun da bulunduğu kareler
    if ok.any():
        center = ((s[ok, 11, :2] + s[ok, 12, :2]) / 2).mean(axis=0)
        width = np.linalg.norm(s[ok, 11, :2] - s[ok, 12, :2], axis=1).mean()
    else:
        center, width = np.array([0.5, 0.5], np.float32), 0.25
    width = max(float(width), 1e-3)
    s[:, :, :2] = (s[:, :, :2] - center) / width
    s[:, :, 2] = s[:, :, 2] / width
    s[missing] = 0.0
    return s.reshape(len(s), -1)


def augment(seq, rng=np.random):
    """Sadece eğitimde: küçük döndürme, ölçek, kaydırma ve gürültü."""
    s = seq.reshape(len(seq), N_PTS, 3).copy()
    missing = (s == 0).all(axis=2)
    a = np.deg2rad(rng.uniform(-10, 10))
    rot = np.array([[np.cos(a), -np.sin(a)], [np.sin(a), np.cos(a)]], np.float32)
    s[:, :, :2] = (s[:, :, :2] @ rot.T) * rng.uniform(0.9, 1.1) + rng.uniform(-0.1, 0.1, 2)
    s += rng.normal(0, 0.01, s.shape).astype(np.float32)
    s[missing] = 0.0
    return s.reshape(len(s), -1)


# Vücuttaki sol/sağ eş noktalar: aynalayınca bunlar yer değiştirmeli.
# (1,4) göz içi, (2,5) göz, (3,6) göz dışı, (7,8) kulak, (9,10) ağız köşesi,
# (11,12) omuz, (13,14) dirsek, (15,16) bilek, (17,18) serçe, (19,20) işaret,
# (21,22) başparmak, (23,24) kalça. 0 (burun) ortada olduğu için yerinde kalır.
ESLER = [(1, 4), (2, 5), (3, 6), (7, 8), (9, 10), (11, 12),
         (13, 14), (15, 16), (17, 18), (19, 20), (21, 22), (23, 24)]


def mirror(seq):
    """Örneği aynalar: sağ elle yapılan işaret, sol elle yapılmış gibi olur.
    normalize() sonrası çağrılmalı (koordinatlar 0 etrafında ortalanmış olmalı)."""
    s = seq.reshape(len(seq), N_PTS, 3).copy()
    s[:, :, 0] = -s[:, :, 0]                       # yatay eksende çevir
    for a, b in ESLER:                             # vücut eşlerini takas et
        s[:, [a, b]] = s[:, [b, a]]
    sol = s[:, 25:46].copy()                       # sol el <-> sağ el
    s[:, 25:46] = s[:, 46:67]
    s[:, 46:67] = sol
    return s.reshape(len(s), -1)


def prepare(seq, train=False, t=48):
    """Ham dizi -> modele girecek (t, 201) dizi. Canlı demoda train=False ile çağrılır."""
    seq = trim(np.asarray(seq, np.float32))
    if train:
        seq = random_time_crop(seq)
    seq = resample(seq, t)
    seq = normalize(seq)
    if train:
        if np.random.random() < 0.5:   # örneklerin yarısı aynalanır -> model el tercihinden bağımsız olur
            seq = mirror(seq)
        seq = augment(seq)
    return seq.astype(np.float32)

In [ ]:
%%writefile models.py
"""Kelime tanıma modeli: kareleri sırayla okuyan küçük bir GRU."""
import torch.nn as nn


class KucukGRU(nn.Module):
    def __init__(self, n_feat=201, n_class=4, hidden=128):
        super().__init__()
        self.inp = nn.Sequential(nn.Linear(n_feat, hidden), nn.LayerNorm(hidden), nn.GELU(), nn.Dropout(0.3))
        self.gru = nn.GRU(hidden, hidden, batch_first=True, bidirectional=True)
        self.head = nn.Sequential(nn.Dropout(0.4), nn.Linear(2 * hidden, n_class))

    def forward(self, x):                 # x: (batch, kare, 201)
        h, _ = self.gru(self.inp(x))      # (batch, kare, 256)
        return self.head(h.mean(dim=1))   # (batch, sınıf sayısı)

### Tek videoda dene ve gözle kontrol et
Yeşil: vücut, mavi: kişinin sol eli, kırmızı: sağ eli. Video aynalı olmadığı için kişinin sol eli görüntünün sağında durur.

In [ ]:
from landmarks import extract_video

ornek = veri[veri.y == 0].iloc[0]
dizi = extract_video(ornek.path)
el_orani = (dizi[:, 75:] != 0).any(axis=1).mean() if len(dizi) else 0.0
print(f"{ornek['sample']} ({SINIF_ADLARI[0]}): {dizi.shape[0]} kare, her kare {dizi.shape[1]} sayı, "
      f"en az bir elin bulunduğu kareler: %{el_orani * 100:.0f}")

k = int((dizi != 0).sum(axis=1).argmax())       # en çok noktanın bulunduğu kare
cap = cv2.VideoCapture(ornek.path)
for _ in range(k + 1):
    ok, kare = cap.read()
cap.release()
h, w = kare.shape[:2]
for j, (x, y, _) in enumerate(dizi[k].reshape(-1, 3)):
    if x or y:
        renk = (0, 255, 0) if j < 25 else (255, 0, 0) if j < 46 else (0, 0, 255)   # BGR
        cv2.circle(kare, (int(x * w), int(y * h)), 4, renk, -1)
plt.figure(figsize=(5, 5))
plt.imshow(cv2.cvtColor(kare, cv2.COLOR_BGR2RGB)); plt.axis("off")
plt.title(f"'{SINIF_ADLARI[0]}' işareti, kare {k}")
plt.savefig("ornek_kare.png", dpi=120, bbox_inches="tight"); plt.show()

### Tüm videoları işle
Birkaç yüz video için genelde 5-15 dakika sürer. Yarıda kalırsa tekrar çalıştır; biten videolar atlanır.

In [ ]:
from concurrent.futures import ProcessPoolExecutor
from tqdm.auto import tqdm
from landmarks import process_one

NOKTA = os.path.join(CIKTI, "noktalar")
os.makedirs(NOKTA, exist_ok=True)
t0 = time.time()
try:
    with ProcessPoolExecutor(max_workers=os.cpu_count() or 2,
                             mp_context=multiprocessing.get_context("spawn")) as ex:
        sonuc = list(tqdm(ex.map(process_one, veri["sample"], veri["path"], [NOKTA] * len(veri), chunksize=2),
                          total=len(veri)))
except Exception as e:
    print("Paralel işlem çalışmadı, videolar tek tek işleniyor:", e)
    sonuc = [process_one(s, p, NOKTA) for s, p in tqdm(list(zip(veri["sample"], veri["path"])))]
hatali = [r for r in sonuc if str(r[1]).startswith("HATA")]
print(f"{len(sonuc)} video {(time.time() - t0) / 60:.1f} dakikada işlendi | hatalı: {len(hatali)}", hatali[:3])

In [ ]:
# Kalite kontrol: her videoda ellerin bulunduğu karelerin oranı
DIZI = {}
for s in veri["sample"]:
    f = os.path.join(NOKTA, s + ".npy")
    if os.path.exists(f):
        d = np.load(f).astype(np.float32)
        if len(d):
            DIZI[s] = d
veri = veri[veri["sample"].isin(DIZI)].copy()
veri["el_orani"] = veri["sample"].map(lambda s: (DIZI[s][:, 75:] != 0).any(axis=1).mean())
print("Ortalama el bulunma oranı (kelimeye göre):")
print(veri.groupby("y").el_orani.mean().rename(index=dict(enumerate(SINIF_ADLARI))).round(2))
dusuk = (veri.split == "train") & (veri.el_orani < 0.1)
print(f"Ellerin neredeyse hiç bulunmadığı {dusuk.sum()} eğitim videosu çıkarıldı (test setine dokunulmaz).")
veri = veri[~dusuk]

## 6. Makine öğrenmesi: modeli eğit

En iyi model, doğrulama setindeki (eğitimde olmayan kişiler) başarıya göre seçilir.

**Aynalama ile veri artırma:** Eğitim örneklerinin yarısı aynalanarak gösterilir, yani sağ elle yapılan
bir işaret sol elle yapılmış gibi de öğretilir. AUTSL'deki 43 işaretçiden yalnızca 2'si solak olduğu için
bu olmadan model yalnızca tek bir eli tanıyor. Bölüm 7'de etkisi ölçülüyor.

In [ ]:
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from preprocess import prepare, mirror
from models import KucukGRU

random.seed(42); np.random.seed(42); torch.manual_seed(42)
cihaz = "cuda" if torch.cuda.is_available() else "cpu"

class IsaretVeri(Dataset):
    def __init__(self, d, egitim, ayna=False):
        self.ornek, self.y, self.egitim, self.ayna = list(d["sample"]), list(d["y"]), egitim, ayna
    def __len__(self):
        return len(self.y)
    def __getitem__(self, i):
        x = prepare(DIZI[self.ornek[i]], train=self.egitim, t=T)
        if self.ayna:                      # testte: her örneği diğer elle yapılmış gibi göster
            x = mirror(x)
        return torch.from_numpy(x), self.y[i]

egit_d, dog_d, test_d = (veri[veri.split == s] for s in ["train", "val", "test"])
egit_yk = DataLoader(IsaretVeri(egit_d, True), batch_size=32, shuffle=True)
dog_yk = DataLoader(IsaretVeri(dog_d, False), batch_size=64)
test_yk = DataLoader(IsaretVeri(test_d, False), batch_size=64)
test_ayna_yk = DataLoader(IsaretVeri(test_d, False, ayna=True), batch_size=64)   # aynalanmış test

@torch.no_grad()
def degerlendir(model, yk):
    model.eval()
    dogru = n = 0
    kayip = 0.0
    for x, y in yk:
        x, y = x.to(cihaz), y.to(cihaz)
        cikti = model(x)
        kayip += nn.functional.cross_entropy(cikti, y, reduction="sum").item()
        dogru += (cikti.argmax(1) == y).sum().item()
        n += len(y)
    return dogru / max(n, 1), kayip / max(n, 1)

C = len(SINIF_ADLARI)
model = KucukGRU(201, C, 128).to(cihaz)
opt = torch.optim.AdamW(model.parameters(), lr=2e-3, weight_decay=0.01)
kayip_fn = nn.CrossEntropyLoss(label_smoothing=0.1)
en_iyi, sabir, gecmis = (-1.0, 0.0), 0, []
for epoch in range(1, EPOCH + 1):
    model.train()
    toplam = 0.0
    for x, y in egit_yk:
        x, y = x.to(cihaz), y.to(cihaz)
        opt.zero_grad()
        kayip = kayip_fn(model(x), y)
        kayip.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()
        toplam += kayip.item() * len(y)
    d_basari, d_kayip = degerlendir(model, dog_yk)
    gecmis.append((epoch, toplam / len(egit_d), d_kayip, d_basari))
    if (d_basari, -d_kayip) > en_iyi:
        en_iyi, sabir = (d_basari, -d_kayip), 0
        torch.save(model.state_dict(), "en_iyi_model.pt")
    else:
        sabir += 1
    if epoch % 5 == 0 or epoch == 1:
        print(f"epoch {epoch:3d} | eğitim kaybı {toplam / len(egit_d):.3f} | doğrulama başarısı %{d_basari * 100:.1f}")
    if sabir >= 25:
        print(f"25 epoch boyunca iyileşme olmadı, epoch {epoch}'te durduruldu.")
        break
print(f"En iyi doğrulama başarısı: %{en_iyi[0] * 100:.1f}")

g = pd.DataFrame(gecmis, columns=["epoch", "egitim_kaybi", "dogrulama_kaybi", "dogrulama_basarisi"])
fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
ax[0].plot(g.epoch, g.egitim_kaybi, label="eğitim"); ax[0].plot(g.epoch, g.dogrulama_kaybi, label="doğrulama")
ax[0].set_title("Kayıp"); ax[0].set_xlabel("epoch"); ax[0].legend()
ax[1].plot(g.epoch, g.dogrulama_basarisi * 100); ax[1].set_title("Doğrulama başarısı (%)"); ax[1].set_xlabel("epoch")
plt.tight_layout(); plt.savefig("egitim_egrisi.png", dpi=120); plt.show()

## 7. Test: modelin hiç görmediği kişiler

In [ ]:
model.load_state_dict(torch.load("en_iyi_model.pt", map_location=cihaz))
model.eval()
tahmin, gercek, guven = [], [], []
with torch.no_grad():
    for x, y in test_yk:
        p = torch.softmax(model(x.to(cihaz)), dim=1).cpu()
        g_, t_ = p.max(dim=1)
        tahmin += t_.tolist(); gercek += y.tolist(); guven += g_.tolist()
tahmin, gercek, guven = np.array(tahmin), np.array(gercek), np.array(guven)
TEST_BASARISI = float((tahmin == gercek).mean())
print(f"TEST BAŞARISI: %{TEST_BASARISI * 100:.1f}  ({len(gercek)} video, {test_d.signer.nunique()} kişi, "
      f"şans eseri tahmin %{100 / C:.0f})")

km = pd.crosstab(pd.Series([SINIF_ADLARI[i] for i in gercek], name="gerçek"),
                 pd.Series([SINIF_ADLARI[i] for i in tahmin], name="tahmin")).reindex(
                 index=SINIF_ADLARI, columns=SINIF_ADLARI, fill_value=0)
print(km)
plt.figure(figsize=(4.5, 4))
plt.imshow(km.values, cmap="Blues")
plt.xticks(range(C), SINIF_ADLARI, rotation=45); plt.yticks(range(C), SINIF_ADLARI)
for i in range(C):
    for j in range(C):
        plt.text(j, i, km.values[i, j], ha="center", va="center")
plt.xlabel("tahmin"); plt.ylabel("gerçek"); plt.title("Karışıklık matrisi (test)")
plt.tight_layout(); plt.savefig("karisiklik_matrisi.png", dpi=120); plt.show()

# --- El tercihinden bağımsızlık ölçümü (aynalama ile veri artırmanın etkisi) ---
ayna_basarisi, _ = degerlendir(model, test_ayna_yk)
print()
print("EL TERCİHİNDEN BAĞIMSIZLIK:")
print(f"  Normal test       : %{TEST_BASARISI * 100:.1f}")
print(f"  Aynalanmış test   : %{ayna_basarisi * 100:.1f}   (diğer elle yapılmış gibi)")
fark = abs(TEST_BASARISI - ayna_basarisi) * 100
print(f"  Fark              : {fark:.1f} puan ->", "iyi, model iki eli de tanıyor" if fark < 10
      else "yüksek, aynalama yeterince öğrenilmemiş")
print()

print("Güven eşiği seçimi (kamera demosu için):")
for esik in [0.4, 0.5, 0.6, 0.7, 0.8]:
    m = guven >= esik
    if m.any():
        print(f"  eşik {esik}: tahminlerin %{m.mean() * 100:.0f}'i kabul edilir, "
              f"kabul edilenlerin başarısı %{(tahmin[m] == gercek[m]).mean() * 100:.1f}")

## 8. Kamera demosunu paketle
Aşağıdaki dosyalar senin bilgisayarında çalışacak demoyu oluşturur.

In [ ]:
%%writefile app.py
"""Canlı işaret tanıma demosu (Kaggle'da eğitilen modelle).

Çalıştırma:   python app.py          (kamera açılmazsa: python app.py 1)
Kısayollar:   SPACE      elle kayıt başlat / bitir
              a          otomatik kaydı aç / kapat
              d          "diğer" sınıfını kapat / aç
              m          aynayı aç / kapat (varsayılan açık; tanımıyorsa kapatmayı dene)
              Backspace  son kelimeyi sil
              c          tüm kelimeleri temizle
              q          çıkış
"""
import os
import sys
import json
import time

import cv2
import numpy as np
import torch
from PIL import Image, ImageDraw, ImageFont

from landmarks import make_landmarkers, to_vector, to_mp_image
from preprocess import prepare
from models import KucukGRU

KLASOR = os.path.dirname(os.path.abspath(__file__))
MODELLER = os.path.join(KLASOR, "models")

with open(os.path.join(MODELLER, "config.json"), encoding="utf-8") as f:
    CFG = json.load(f)
with open(os.path.join(MODELLER, "labels.json"), encoding="utf-8") as f:
    ETIKETLER = json.load(f)

ESIK = CFG["esik"]            # bu güvenin altındaki tahminler kabul edilmez
DIGER = CFG["diger_index"]    # "diğer" sınıfının numarası (yoksa None)
BITIS_BOSLUK = 12             # el bu kadar kare görünmezse kelime bitti sayılır (~0,4 sn)
MIN_KARE = 10                 # bundan kısa kayıtlar gürültü sayılır
MAX_KARE = 150                # bir kelime en fazla ~5 sn
KAMERA = int(sys.argv[1]) if len(sys.argv) > 1 else 0

model = KucukGRU(CFG["n_feat"], CFG["n_class"], CFG["hidden"])
model.load_state_dict(torch.load(os.path.join(MODELLER, "model.pt"), map_location="cpu"))
model.eval()
hand_lm, pose_lm = make_landmarkers(os.path.join(MODELLER, "hand_landmarker.task"),
                                    os.path.join(MODELLER, "pose_landmarker_full.task"))


def yazi_tipi(boyut):
    """Türkçe karakterleri (ğ, ş, ı, İ) çizebilen bir yazı tipi bulur."""
    adaylar = ["arial.ttf", "C:/Windows/Fonts/arial.ttf",
               "/System/Library/Fonts/Supplemental/Arial.ttf", "/Library/Fonts/Arial.ttf",
               "/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf"]
    for yol in adaylar:
        try:
            return ImageFont.truetype(yol, boyut)
        except OSError:
            pass
    try:
        return ImageFont.load_default(size=boyut)
    except TypeError:
        return ImageFont.load_default()


BUYUK, KUCUK = yazi_tipi(24), yazi_tipi(17)


DIGER_KAPALI = False          # d tuşu: "diğer" sınıfını yok sayıp en olası kelimeyi seçer
# AUTSL videolarındaki aktif el "sol el" kutusuna düşüyor (videolar aynalanmış kaydedilmiş).
# Bu yüzden kamera görüntüsünü de aynalayıp veriyoruz; varsayılan AÇIK. m tuşu kapatır.
# Model aynalama ile veri artırma kullanılarak yeniden eğitilirse buna gerek kalmayacak.
AYNA = bool(CFG.get("ayna", True))
son_tahminler = ""            # ekranda gösterilen ilk 3 tahmin


def tahmin_et(kareler):
    """Kaydedilen kareleri modele verir; her sınıfın olasılığını döndürür."""
    x = torch.from_numpy(prepare(np.array(kareler, np.float32), train=False, t=CFG["T"]))[None]
    with torch.no_grad():
        p = torch.softmax(model(x), dim=1)[0].numpy()
    if DIGER_KAPALI and DIGER is not None:
        p[DIGER] = 0.0
        p = p / p.sum()
    return p


def kelimeyi_bitir(kareler):
    """Kaydı değerlendirir; (eklenecek kelime ya da None, ekrana yazılacak bilgi) döndürür."""
    global son_tahminler
    if len(kareler) < MIN_KARE:
        return None, "Kayıt çok kısaydı, tekrar dene"
    p = tahmin_et(kareler)
    sira = np.argsort(p)[::-1][:3]
    son_tahminler = "Model: " + "  ".join(f"{ETIKETLER[str(i)]} %{p[i] * 100:.0f}" for i in sira)
    print(son_tahminler)
    idx, guven = int(sira[0]), float(p[sira[0]])
    ad = ETIKETLER[str(idx)]
    if idx == DIGER:
        return None, f"Tanınmadı (diğer, {guven:.2f}) - tekrar dene"
    if guven < ESIK:
        return None, f"Emin olamadım ({ad}? {guven:.2f}) - tekrar dene"
    return ad, f"Tanındı: {ad} ({guven:.2f})"


def kare_kirp(kare):
    """Eğitim videoları kare biçiminde (512x512) olduğu için kamera görüntüsü de ortadan kare kırpılır.
    Aksi halde 4:3 görüntüde yatay ve dikey koordinatlar farklı ölçeklenir ve model şaşırır."""
    h, w = kare.shape[:2]
    m = min(h, w)
    y0, x0 = (h - m) // 2, (w - m) // 2
    return kare[y0:y0 + m, x0:x0 + m].copy()


def kadraj_uyarisi(vek):
    """Omuzlar kadrajda değilse uyarı döndürür (eğitim videolarında omuzlar hep görünür)."""
    v = vek.reshape(-1, 3)
    if not v[:25].any():
        return "Vücudun görünmüyor: kameranın karşısına geç"
    omuzlar = v[[11, 12], :2]
    if (omuzlar < 0.03).any() or (omuzlar > 0.97).any():
        return "Omuzların kadrajda değil: biraz geri çekil"
    if np.linalg.norm(omuzlar[0] - omuzlar[1]) > 0.6:
        return "Kameraya çok yakınsın: biraz geri çekil"
    return ""


# MediaPipe'ın resmi bağlantı listeleri: hangi nokta hangisine bağlanır (iskeleti çizmek için).
EL_BAGLANTI = [(0, 1), (1, 2), (2, 3), (3, 4), (0, 5), (5, 6), (6, 7), (7, 8),
               (5, 9), (9, 10), (10, 11), (11, 12), (9, 13), (13, 14), (14, 15), (15, 16),
               (13, 17), (17, 18), (18, 19), (19, 20), (0, 17)]
VUCUT_BAGLANTI = [(11, 12), (11, 13), (13, 15), (12, 14), (14, 16),          # omuz - dirsek - bilek
                  (11, 23), (12, 24), (23, 24),                              # gövde
                  (15, 17), (15, 19), (15, 21), (17, 19),                    # sol el (vücut modelinin gördüğü)
                  (16, 18), (16, 20), (16, 22), (18, 20),                    # sağ el
                  (0, 2), (2, 7), (0, 5), (5, 8), (9, 10)]                   # yüz


def cizgileri_ciz(kare, vek):
    """Noktalar arasına iskelet çizgilerini çizer. Model bu çizgileri kullanmaz, sadece gösterim içindir."""
    h, w = kare.shape[:2]
    v = vek.reshape(-1, 3)

    def ciz(baslangic, baglantilar, renk):
        for a, b in baglantilar:
            pa, pb = v[baslangic + a], v[baslangic + b]
            if (pa[0] or pa[1]) and (pb[0] or pb[1]):        # iki ucu da bulunduysa çiz
                cv2.line(kare, (int(pa[0] * w), int(pa[1] * h)),
                         (int(pb[0] * w), int(pb[1] * h)), renk, 2, cv2.LINE_AA)

    ciz(0, VUCUT_BAGLANTI, (0, 170, 0))      # vücut: yeşil
    ciz(25, EL_BAGLANTI, (200, 60, 60))      # sol el: mavi
    ciz(46, EL_BAGLANTI, (60, 60, 200))      # sağ el: kırmızı


def noktalari_ciz(kare, vek):
    h, w = kare.shape[:2]
    for j, (x, y, _) in enumerate(vek.reshape(-1, 3)):
        if x or y:
            renk = (0, 255, 0) if j < 25 else (255, 0, 0) if j < 46 else (0, 0, 255)   # BGR
            cv2.circle(kare, (int(x * w), int(y * h)), 3, renk, -1)


def yazilari_ciz(kare, satirlar, alt_satir):
    """cv2.putText Türkçe harfleri çizemediği için yazılar PIL ile çizilir."""
    img = Image.fromarray(cv2.cvtColor(kare, cv2.COLOR_BGR2RGB))
    d = ImageDraw.Draw(img)
    for k, (metin, renk) in enumerate(satirlar):
        d.text((12, 10 + 32 * k), metin, font=BUYUK, fill=renk, stroke_width=3, stroke_fill=(0, 0, 0))
    d.text((12, img.height - 28), alt_satir, font=KUCUK, fill=(255, 255, 255), stroke_width=2, stroke_fill=(0, 0, 0))
    return cv2.cvtColor(np.array(img), cv2.COLOR_RGB2BGR)


cap = cv2.VideoCapture(KAMERA)
if not cap.isOpened():
    sys.exit(f"Kamera {KAMERA} açılamadı. Kamerayı kullanan başka bir uygulamayı kapat veya 'python app.py 1' dene.")

baslangic, son_ts = time.monotonic(), -1
durum, tampon, bosluk, elle = "BEKLE", [], 0, False
otomatik = True
kelimeler, bilgi = [], "Elini kaldır ve bir işaret yap"
fps, onceki = 0.0, time.monotonic()
print("Demo başladı. Çıkmak için pencere seçiliyken q tuşuna bas.")

while True:
    ok, kare = cap.read()
    if not ok:
        print("Kameradan görüntü alınamadı.")
        break
    kare = kare_kirp(kare)
    if AYNA:
        kare = cv2.flip(kare, 1)      # modele giden kareyi ters cevir
    ts = max(int((time.monotonic() - baslangic) * 1000), son_ts + 1)   # zaman damgası hep artmalı
    son_ts = ts
    img = to_mp_image(kare)                               # modele AYNALANMAMIŞ kare gider
    vek = to_vector(pose_lm.detect_for_video(img, ts), hand_lm.detect_for_video(img, ts))
    el_var = bool((vek[75:] != 0).any())
    sol_var, sag_var = bool((vek[75:138] != 0).any()), bool((vek[138:] != 0).any())
    aktif_el = "iki el" if sol_var and sag_var else "SOL" if sol_var else "SAĞ" if sag_var else "-"

    # Otomatik mod: el görününce kayıt başlar, el ~0,4 sn kaybolunca kelime biter.
    if durum == "BEKLE" and otomatik and el_var:
        durum, tampon, bosluk, elle = "KAYIT", [], 0, False
    if durum == "KAYIT":
        tampon.append(vek)
        bosluk = 0 if el_var else bosluk + 1
        if (not elle and bosluk >= BITIS_BOSLUK) or len(tampon) >= MAX_KARE:
            kelime, bilgi = kelimeyi_bitir(tampon[:len(tampon) - bosluk])
            if kelime:
                kelimeler.append(kelime)
            durum, elle = "BEKLE", False

    simdi = time.monotonic()
    fps = 0.9 * fps + 0.1 / max(simdi - onceki, 1e-6)
    onceki = simdi

    cizgileri_ciz(kare, vek)
    noktalari_ciz(kare, vek)
    ekran = cv2.resize(cv2.flip(kare, 1), (640, 640))     # sadece ekranda ayna görüntüsü
    if durum == "KAYIT":
        cv2.circle(ekran, (ekran.shape[1] - 28, 28), 12, (0, 0, 255), -1)
    satirlar = [
        (f"{durum}{' (elle)' if elle else ''} | Aktif el kutusu: {aktif_el}"
         f" | Ayna: {'AÇIK' if AYNA else 'kapalı'}", (255, 255, 255)),
        (f"Oto: {'açık' if otomatik else 'kapalı'} | Diğer: {'kapalı' if DIGER_KAPALI else 'açık'}"
         f" | FPS {fps:.0f}", (200, 200, 200)),
        (bilgi, (255, 230, 0)),
        (son_tahminler, (150, 220, 255)),
        ("Kelimeler: " + " ".join(kelimeler), (120, 255, 120)),
        (kadraj_uyarisi(vek), (255, 90, 90)),
    ]
    ekran = yazilari_ciz(ekran, satirlar, "SPACE: kayıt  a: oto  m: ayna  d: diğer  Backspace: sil  c: temizle  q: çıkış")
    cv2.imshow("Isaret Tanima", ekran)

    tus = cv2.waitKey(1) & 0xFF
    if tus == ord("q"):
        break
    elif tus == 32:                                       # SPACE: elle kayıt
        if durum == "KAYIT":
            kelime, bilgi = kelimeyi_bitir(tampon)
            if kelime:
                kelimeler.append(kelime)
            durum, elle = "BEKLE", False
        else:
            durum, tampon, bosluk, elle = "KAYIT", [], 0, True
            bilgi = "Elle kayıt: işareti yap, bitince SPACE"
    elif tus == ord("a"):
        otomatik = not otomatik
    elif tus == ord("m"):
        AYNA = not AYNA
        bilgi = "Ayna " + ("açıldı" if AYNA else "kapandı") + ": sol/sağ el ataması değişti"
    elif tus == ord("d") and DIGER is not None:
        DIGER_KAPALI = not DIGER_KAPALI
        bilgi = "Diğer sınıfı " + ("kapatıldı: hep en olası kelime seçilir" if DIGER_KAPALI else "açıldı")
    elif tus in (8, 127) and kelimeler:                   # Backspace (Windows: 8, macOS: 127)
        kelimeler.pop()
    elif tus == ord("c"):
        kelimeler, bilgi = [], "Temizlendi"

cap.release()
hand_lm.close()
pose_lm.close()
cv2.destroyAllWindows()

In [ ]:
%%writefile requirements.txt
mediapipe==0.10.32
opencv-python
numpy
pillow>=10.1
torch

In [ ]:
%%writefile BENIOKU.txt
İŞARET TANIMA DEMOSU - KENDİ BİLGİSAYARINDA ÇALIŞTIRMA
=====================================================

Bu klasördekiler:
  app.py            kamera demosu (bunu çalıştıracaksın)
  landmarks.py      görüntüden el ve vücut noktalarını çıkarır (MediaPipe)
  preprocess.py     noktaları modele hazırlar (eğitimdekiyle aynı dosya)
  models.py         kelime tanıma modeli (GRU)
  models/           eğitilmiş model, kelime listesi ve MediaPipe model dosyaları
  ornek_videolar/   öğretilen her kelimenin örnek videoları (GIF; çift tıklayıp izle, işareti buradan çalış)
  sonuclar/         hocaya gösterebileceğin grafikler (örnek kare, eğitim eğrisi, karışıklık matrisi)

1) KURULUM (bir kez, internet gerekir; torch indirmesi birkaç dakika sürebilir)
   - Python 3.11 veya 3.12 kurulu olmalı (python.org).
     Windows'ta kurarken "Add python.exe to PATH" kutusunu işaretle.
   - Bu klasörde bir terminal aç:
       Windows: klasörün adres çubuğuna cmd yazıp Enter'a bas.
       macOS:   Terminal'de  cd  yazıp bir boşluk bırak, klasörü pencereye sürükle, Enter.
   - Sırayla şu komutları yaz:
       python -m venv venv
       venv\Scripts\activate          (macOS: source venv/bin/activate)
       pip install -r requirements.txt

2) ÇALIŞTIRMA
       venv\Scripts\activate          (macOS: source venv/bin/activate)
       python app.py
   Kamera açılmazsa:  python app.py 1
   macOS'ta ilk seferde Terminal'e kamera izni vermen gerekebilir.

3) KULLANIM
   - Belden yukarın kadrajda olsun, ışık önden gelsin.
   - Görüntü ortadan kare kırpılır (eğitim videoları kare olduğu için); kadrajın ortasında dur.
   - Ellerin aşağıda (kadraj dışında) beklesin.
   - Elini kaldır, işareti yap, ellerini tekrar indir. Kelime ekranda listeye eklenir.
   - Otomatik mod sorun çıkarırsa SPACE'e bas, işareti yap, tekrar SPACE'e bas.
   - Backspace: son kelimeyi sil   c: temizle   a: otomatik modu aç/kapat   q: çıkış
   - "Emin olamadım" yazıyorsa işareti biraz daha net ve AUTSL videolarındaki gibi yap.

4) SIK KARŞILAŞILAN SORUNLAR
   - "python bulunamadı": Python'u PATH seçeneğiyle yeniden kur ya da "py" komutunu dene.
   - "No module named ...": venv etkin değil; activate komutunu tekrar çalıştır.
   - mediapipe kurulmuyorsa: pip install mediapipe   (sürüm belirtmeden)

In [ ]:
PAKET = os.path.join(CIKTI, "isaret_demo")
shutil.rmtree(PAKET, ignore_errors=True)
os.makedirs(os.path.join(PAKET, "models"))
os.makedirs(os.path.join(PAKET, "sonuclar"))

torch.save(model.cpu().state_dict(), os.path.join(PAKET, "models", "model.pt"))
with open(os.path.join(PAKET, "models", "labels.json"), "w", encoding="utf-8") as f:
    json.dump({str(i): ad for i, ad in enumerate(SINIF_ADLARI)}, f, ensure_ascii=False, indent=2)
with open(os.path.join(PAKET, "models", "config.json"), "w", encoding="utf-8") as f:
    json.dump({"T": T, "n_feat": 201, "n_class": C, "hidden": 128, "esik": ESIK,
               "diger_index": C - 1 if DIGER_SINIFI else None,
               "ayna": False,   # model aynalama ile eğitildi; demo artık görüntüyü çevirmiyor
               "ayna_test_basarisi": round(ayna_basarisi, 4),
               "test_basarisi": round(TEST_BASARISI, 4)}, f, indent=2)
for ad in MODEL_URL:
    shutil.copy(ad, os.path.join(PAKET, "models"))
for ad in ["app.py", "landmarks.py", "preprocess.py", "models.py", "requirements.txt", "BENIOKU.txt"]:
    shutil.copy(ad, PAKET)
for ad in ["ornek_kare.png", "egitim_egrisi.png", "karisiklik_matrisi.png"]:
    if os.path.exists(ad):
        shutil.copy(ad, os.path.join(PAKET, "sonuclar"))
if os.path.isdir("ornek_videolar"):
    shutil.copytree("ornek_videolar", os.path.join(PAKET, "ornek_videolar"))

zip_yolu = shutil.make_archive(os.path.join(CIKTI, "demo_paketi"), "zip", CIKTI, "isaret_demo")
print("Hazır:", zip_yolu, round(os.path.getsize(zip_yolu) / 1e6, 1), "MB")
print("Kelimeler:", SINIF_ADLARI, "| test başarısı: %" + f"{TEST_BASARISI * 100:.1f}")

## 9. Sonraki adım: kendi bilgisayarında kamera demosu

1. Sağdaki *Output* bölümünden `demo_paketi.zip` dosyasını indir ve bir klasöre çıkar.
2. `isaret_demo/BENIOKU.txt` dosyasındaki adımları izle: sanal ortam kur, `pip install -r requirements.txt`, sonra `python app.py`.
3. Elini kaldır, işareti yap, ellerini indir; tanınan kelime ekranda listeye eklenir.

Kelime değiştirmek istersen 1. hücredeki `SECILEN_KELIMELER` listesini değiştirip tekrar **Run All** de.